# 10 — The A2A Protocol

Modules 01 to 03 built agents that all lived in one process. They shared a state
object, a framework, and a codebase.

A2A is for when they do not.

```
your agent  ──▶  a different agent, on a different server,
                 written by a different team, in a different framework
```

The two sides share **nothing** — no memory, no prompts, no tools. All that
crosses the boundary is a card and some messages. That constraint is what makes
it work between teams, and also what makes it more expensive than a function
call.

## First, do you need it?

Usually not.

If your workflow is deterministic — you know which tools run in what order — a
function call beats an HTTP round trip, a task lifecycle and a card fetch, every
time.

A2A earns its cost in one situation: **the agents are deployed separately, by
different teams, and you do not control both sides.** That is when you need a
contract instead of an import.

## MCP or A2A?

You met MCP in module 02. They are not competitors.

| | MCP | A2A |
|---|---|---|
| Connects | your agent to its tools | your agent to another agent |
| The other side is | a function you call | a system that thinks |
| You control it | yes | often not |

Production systems run both.

| § | What |
|---|---|
| 0 | Start three agents |
| 1 | The Agent Card — discovery |
| 2 | The wire — JSON-RPC, seen raw |
| 3 | Tasks and states |
| 4 | Streaming, and the queue underneath it |
| 5 | Dropping the connection |
| 6 | `input-required` |
| 7 | One agent calling another |

---
## 0. Start the agents

In [ ]:
%pip install -q aiohttp

In [ ]:
import asyncio
import importlib.util
import json
import sys
from pathlib import Path

from aiohttp import web

sys.path.insert(0, str(Path.cwd()))
from a2a_mini import A2AClient


def load(path, name):
    """Import an agent module by path, so the notebook can host it in-process."""
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


echo_mod = load("agents/echo_agent.py", "echo_agent")
counter_mod = load("agents/counter_agent.py", "counter_agent")
planner_mod = load("agents/planner_agent.py", "planner_agent")

runners = []


async def serve(server, port):
    runner = web.AppRunner(server.app())
    await runner.setup()
    await web.TCPSite(runner, "127.0.0.1", port).start()
    runners.append(runner)


# Normally each of these is `python agents/echo_agent.py` in its own terminal.
# Running them here keeps everything in one notebook.
await serve(echo_mod.server, 9101)
await serve(counter_mod.server, 9102)
await serve(planner_mod.server, 9103)
print("three agents listening on 9101, 9102, 9103")

---
## 1. The Agent Card

A JSON file at a fixed path. One GET and you know what the agent does, where to
send work, and whether you may stream to it.

No registry, no SDK, no documentation. That is the point of the format — a
stranger's agent is usable without anyone arranging anything first.

In [ ]:
import urllib.request

raw = urllib.request.urlopen(
    "http://127.0.0.1:9102/.well-known/agent-card.json").read()
print(json.dumps(json.loads(raw), indent=2))

`capabilities.streaming` is worth noticing. It is **declared**, not discovered.

A client that streams to an agent which cannot stream has made a protocol error,
not a slow request. Putting it in the card means the client can check before it
tries, rather than parsing an error afterwards.

In [ ]:
for port in (9101, 9102, 9103):
    card = await A2AClient(f"http://127.0.0.1:{port}").discover()
    streams = "streams" if card.capabilities.get("streaming") else "no streaming"
    print(f"{card.name:<9} {streams:<13} {card.skills[0].name}")

---
## 2. The wire

JSON-RPC 2.0 over HTTP POST. Nothing custom.

Look at this before touching any client library — the protocol is small enough
to read, and knowing that changes how you debug it.

In [ ]:
request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": "hello a2a"}],
        }
    },
}

body = json.dumps(request).encode()
response = urllib.request.urlopen(urllib.request.Request(
    "http://127.0.0.1:9101", data=body,
    headers={"Content-Type": "application/json"}))
print(json.dumps(json.loads(response.read()), indent=2))

Two things in that JSON.

**`role` is `"user"`** even though a program sent it. In A2A, "user" means the
side asking and "agent" means the side answering, whatever is actually at each
end. A supervisor calling a specialist sends `role: "user"`.

**The result is a Task, not an answer.** It has an id, a state, a history and a
list of artifacts. Even for work that finished instantly.

---
## 3. Tasks and states

A task is not a request and a response. It is an object with a lifecycle.

```
submitted ──▶ working ──▶ completed
                 │
                 ├──▶ input-required ──▶ working ──▶ completed
                 ├──▶ failed
                 └──▶ canceled
```

The reason is simple: real work takes longer than a sensible HTTP timeout. Once
you accept that, the task needs an id — so you can ask about it later, from
somewhere else.

In [ ]:
echo = A2AClient("http://127.0.0.1:9101")
await echo.discover()

task = await echo.send("protocols are just agreements")
print(f"id       {task['id']}")
print(f"state    {task['state']}")
print(f"history  {len(task['history'])} messages")
print(f"artifact {task['artifacts'][0]['parts'][0]['text']}")

---
## 4. Streaming, and the queue underneath it

`message/send` holds the HTTP connection open until the work finishes. Fine for
the echo agent. Useless for anything slow — a long task hits a proxy timeout and
you lose a result the server produced successfully.

So: `message/stream`, which returns Server-Sent Events.

### But look at what is actually happening on the server

The agent is producing events while a client may or may not be listening. So the
events cannot be written straight to an HTTP response. They go into a **queue**,
and whoever is listening drains it.

```
executor  ──put()──▶  EventQueue  ──subscribe()──▶  SSE response
                          │
                          ├── replay buffer   every event so far
                          └── subscribers     one queue each
```

This is the part most A2A material skips, and it is the part that matters.

In [ ]:
counter = A2AClient("http://127.0.0.1:9102")
await counter.discover()

async for event in counter.stream("count to 4"):
    kind = event.get("kind")
    if kind == "progress":
        print(f"progress  {event['step']}/{event['of']}")
    elif kind == "status-update":
        print(f"status    {event['state']}")
    elif kind == "artifact-update":
        print(f"artifact  {event['artifact']['parts'][0]['text']}")

In [ ]:
# The agent's own code. Notice what is missing from it.
import inspect
print(inspect.getsource(counter_mod.execute))

`execute()` has no reference to an HTTP response and never checks whether anyone
is connected. It calls `queue.put()` and moves on.

That separation is the whole design. Wire an agent directly to a response object
instead, and a dropped connection kills work that was running perfectly well.

---
## 5. Dropping the connection

Here is the payoff.

Start a long count, walk away after three steps, wait, then reattach with
`tasks/resubscribe`.

Watch for two things: the task does not pause while nobody is listening, and it
does not restart when someone comes back.

In [ ]:
task_id = None
before = []

async for event in counter.stream("count to 8"):
    task_id = event.get("taskId")
    if event.get("kind") == "progress":
        before.append(event["step"])
        print(f"watching   step {event['step']}")
        if len(before) == 3:
            print("...disconnecting")
            break

In [ ]:
print("away for 3 seconds — the server has zero subscribers")
await asyncio.sleep(3)

state = await counter.get(task_id)
print(f"polled     task is '{state['state']}' — it never paused")

In [ ]:
after = []

async for event in counter.resubscribe(task_id):
    if event.get("kind") == "progress":
        after.append(event["step"])
        tag = "replayed" if event["step"] in before else "live"
        print(f"  step {event['step']}   {tag}")
    if event.get("final"):
        break

print(f"\nbefore  {before}")
print(f"after   {after}")
print(f"no gap, no restart: {after == list(range(1, 9))}")

The replay buffer is why the first three arrive instantly and the rest arrive one
per second. You reattached at step 6 or 7, got the history, then continued live.

Without the queue, none of this is possible: no reattaching, no second watcher,
and a dropped connection means lost work.

In [ ]:
# The queue for that task, after the fact.
queue = counter_mod.server.queues.get(task_id)
print(f"events recorded  {queue.event_count}")
print(f"subscribers now  {queue.subscriber_count}")

---
## 6. `input-required`

The state people forget, and the one that separates a task from a function call.

A function that needs more input has to fail and be called again — losing
everything it had. A task **pauses**, keeps its history, and continues when the
answer arrives. An hour later, from a different process.

In [ ]:
planner = A2AClient("http://127.0.0.1:9103")
await planner.discover()

trip_id = None
async for event in planner.stream("plan a trip to Kyoto"):
    if event.get("kind") == "status-update":
        trip_id = event["taskId"]
        print(f"status {event['state']:<16} {event.get('note', '')}")
        if event["state"] == "input-required":
            break

In [ ]:
# Answer on the SAME task id. The destination was never re-sent — it is still
# on the server, attached to this task.
async for event in planner.stream("3 days", task_id=trip_id):
    if event.get("kind") == "artifact-update":
        print(event["artifact"]["parts"][0]["text"])
    if event.get("final"):
        break

final = await planner.get(trip_id)
print(f"\nstate    {final['state']}")
print(f"history  {len(final['history'])} messages — both sides, one task")

---
## 7. One agent calling another

Everything so far had a notebook as the client. Now an agent is the client.

This is the shape of a supervisor: it discovers what is available, picks one, and
delegates. It cannot see inside the agent it calls — only the card told it what
that agent does, and only the artifact tells it what came back.

In [ ]:
async def supervisor(question: str) -> str:
    """Pick an agent by reading cards, delegate, return its artifact.

    Routing is a judgement, so a real supervisor would use a model here. The
    point of this version is what it CANNOT do: it has no access to either
    agent's memory, prompt or tools. It reads cards and it sends messages.
    """
    fleet = []
    for port in (9101, 9102, 9103):
        client = A2AClient(f"http://127.0.0.1:{port}")
        fleet.append((await client.discover(), client))

    for card, client in fleet:
        if any(word in question.lower()
               for word in card.skills[0].name.lower().split()):
            print(f"routing to '{card.name}' — its card says: {card.skills[0].description}")
            task = await client.send(question)
            if task["artifacts"]:
                return task["artifacts"][0]["parts"][0]["text"]
            return f"(no artifact, task {task['state']})"

    return "no agent advertises a matching skill"


print(await supervisor("reverse this sentence"))

---
## 8. Whose identity is the call made with?

Everything above ran with no auth at all. Two lines about what changes when you
add it, because this is the decision people get wrong.

An agent calls another agent on behalf of **a person**. So when the supervisor
forwards work, there are two choices:

**Mint a service token.** The supervisor authenticates as itself. Simple, and
wrong: every downstream permission check now sees "the supervisor", which has to
be allowed to do everything any user might do. One user's access becomes every
user's access, and nothing looks broken until an audit.

**Forward the caller's token.** The supervisor passes down the same credential
it received. Each specialist checks the *person's* groups, so a user who cannot
see a dataset still cannot see it two hops away.

The second is right, and it is not the default in any framework — you have to
write it.

The Agent Card is where this is declared:

```json
"securitySchemes": {
  "oauth2": { "type": "oauth2", "flows": { ... } }
}
```

A client reads that before its first call, the same way it reads `streaming`.

**Never log the token.** It is a live credential with a real person's
permissions attached, and logs get shipped somewhere.

---
## What you would add for production

**Auth.** The card declares a security scheme; this build ignores it. Real
deployments use OAuth 2.0 and check on every call.

**Push notifications.** For work measured in hours, a webhook beats holding a
stream open.

**A durable queue.** Ours is a Python list in memory. Restart the process and
every running task is lost. Redis or a database, in production.

**Signed cards.** v1.0 added cryptographic signing, so you can verify a card
came from who it claims.

## What to remember

**A task outlives its connection.** Everything else — the id, the queue, the
states, resubscribe — follows from that one fact.

**The two sides share nothing.** Only the card and the messages cross. That is
why it works between teams, and why it costs more than an import.

**Most systems do not need it.** One team, one repo, deterministic flow: use
function calls. A2A is for when you cannot.

In [ ]:
for runner in runners:
    await runner.cleanup()
print("agents stopped")